In [2]:
import pandas as pd

In [3]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [5]:
row = df.iloc[0]
row

PULocationID                             43
DOLocationID                            186
trip_distance                          1.68
total_amount                          22.15
tpep_pickup_datetime    2025-11-01 00:13:25
Name: 0, dtype: object

In [6]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [7]:
def ride_from_row(row):
    return Ride(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
    )

In [8]:
ride = ride_from_row(df.iloc[0])
ride
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [9]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [11]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()